# Moshi Family Test

포크 `latentforge/transformers-moshi` (5.15.0.dev0) 기준.

`MoshiProcessor`는 feature extractor / tokenizer / **Mimi 코덱**을 한 번에 묶습니다.
일반 오디오 모델과 달리 `input_values`를 내지 않고, 스트림별로 **이미 인코딩된 코드**를 냅니다:

- `audio=` → `user_audio_codes` (사용자 발화)
- `assistant_audio=` → `assistant_audio_codes` (Moshi 자신의 발화, 프롬프트용)
- `text=` → `input_ids` (오디오 프레임 수에 맞춰 자동 pad)

In [ ]:
import transformers

print(transformers.__version__)
print([n for n in dir(transformers) if "Moshi" in n or "Mimi" in n])

## 1. 프로세서 조립

`MoshiProcessor(feature_extractor, tokenizer, audio_tokenizer, num_codebooks=8)`.
`audio_tokenizer`는 `MimiModel` 인스턴스이고, `num_codebooks`는 코덱이 아니라 **Moshi 체크포인트**의 속성입니다
(Mimi는 Moshi가 읽는 것보다 많은 코드북을 낼 수 있음).

In [ ]:
import torch
from transformers import (
    AutoFeatureExtractor,
    AutoTokenizer,
    MimiModel,
    MoshiForConditionalGeneration,
    MoshiProcessor,
)

MODEL_ID = "kyutai/moshiko-pytorch-bf16"
MIMI_ID = "kyutai/mimi"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

mimi = MimiModel.from_pretrained(MIMI_ID).to(device).eval()
processor = MoshiProcessor(
    feature_extractor=AutoFeatureExtractor.from_pretrained(MIMI_ID),
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID),
    audio_tokenizer=mimi,
    num_codebooks=8,
)

model = MoshiForConditionalGeneration.from_pretrained(MODEL_ID, dtype=dtype).to(device).eval()

# 저장해두면 다음부터 MoshiProcessor.from_pretrained(...) 한 줄로 로드된다.
# processor.save_pretrained("./moshi-processor")

sr = mimi.config.sampling_rate
frame_rate = mimi.config.frame_rate
print(f"device={device} dtype={dtype} sr={sr} frame_rate={frame_rate}")

## 2. 입력 인코딩

텍스트는 오디오 프레임 수에 맞춰 자동으로 오른쪽 pad 됩니다
(텍스트가 더 길면 `ValueError` — 오디오를 늘리거나 텍스트를 줄여야 함).

In [ ]:
import numpy as np

# 실제 파일: librosa.load("user.wav", sr=sr, mono=True)[0]
seconds = 5.0
user_audio = np.zeros(int(sr * seconds), dtype=np.float32)

inputs = processor(text="Hello", audio=user_audio, sampling_rate=sr)

for k, v in inputs.items():
    print(f"{k:24s} {tuple(v.shape)}")

## 3. 생성

프로세서 출력 키(`input_ids`, `user_audio_codes`, `assistant_audio_codes`)가
`generate`의 인자명과 그대로 맞아서 `**inputs`로 바로 넘어갑니다.

In [ ]:
inputs = inputs.to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=True,
        temperature=0.8,
        top_k=250,
        return_audio_codes=True,
    )

text_out = processor.tokenizer.decode(output.sequences[0], skip_special_tokens=True)
print("text:", text_out)
print("audio_codes:", tuple(output.audio_codes.shape))

In [ ]:
from IPython.display import Audio

# 코드 -> 파형은 프로세서가 담당 (내부적으로 Mimi decode)
waveform = processor.decode_audio(output.audio_codes)
print(tuple(waveform.shape), waveform.shape[-1] / sr, "sec")

Audio(waveform[0, 0].float().cpu().numpy(), rate=sr)

## 4. 라이브 사용자 없이 생성 — `get_silence_audio_codes`

Moshi는 full-duplex라 매 스텝 user 프레임을 소비합니다. 사용자가 없으면 무음을 넣어야 하는데,
Mimi는 스트리밍 코덱이라 **무음이 상수 코드로 양자화되지 않습니다** (컨볼루션 상태가 프레임을 넘어 이어짐).
한 프레임을 반복하면 안 되고, 구간 전체를 한 번에 인코딩해야 해서 이 헬퍼가 있습니다.

In [ ]:
num_frames = 100
silence_codes = processor.get_silence_audio_codes(num_frames=num_frames, batch_size=1)
print("silence:", tuple(silence_codes.shape))  # (B, num_codebooks, num_frames)

# 한 프레임 반복 != 실제 무음 인코딩 (위 설명 확인)
print("frame 0 == frame 50?", torch.equal(silence_codes[..., 0], silence_codes[..., 50]))

with torch.no_grad():
    out = model.generate(user_audio_codes=silence_codes, max_new_tokens=32, do_sample=True)
print(processor.tokenizer.decode(out.sequences[0], skip_special_tokens=True))

## 5. 양쪽 스트림 프롬프트

`assistant_audio`로 "Moshi가 이미 말한 내용"을 함께 주면 대화 중간부터 이어갈 수 있습니다.

In [ ]:
assistant_audio = np.zeros(int(sr * seconds), dtype=np.float32)  # 실제로는 Moshi의 이전 발화

duplex = processor(
    text="Hi there",
    audio=user_audio,
    assistant_audio=assistant_audio,
    sampling_rate=sr,
).to(device)

print(sorted(duplex.keys()))
print({k: tuple(v.shape) for k, v in duplex.items()})

## 6. 스트리밍 대화 (chunk-by-chunk full-duplex)

**주의:** Moshi의 `generate`에는 내장 스트리밍/스텝 API가 없습니다 (`streamer` 인자 없음).
실시간 대화는 아래 세 가지를 직접 이어붙여 구현합니다:

| 상태 | 무엇 | 왜 필요한가 |
|---|---|---|
| `padding_cache` | `MimiConv1dPaddingCache` | Mimi causal conv의 프레임 간 패딩 상태. 없으면 청크 경계마다 클릭 노이즈 |
| `encoder_past_key_values` | Mimi encoder transformer KV | 코덱 인코더의 시간축 문맥 |
| `past_key_values` | Moshi temporal transformer KV | 대화 문맥. 이게 없으면 매 청크가 새 대화 |

그리고 2번째 청크부터는 **`concat_unconditional_inputs=False`** — 안 그러면 매 청크마다
unconditional BOS 프레임이 다시 붙습니다.

`use_streaming=True`로 부르면 Mimi가 `padding_cache`를 알아서 만들어 반환하므로,
첫 청크는 `None`으로 넘기고 이후엔 받은 걸 그대로 되먹이면 됩니다.

In [ ]:
class MoshiStreamingSession:
    """청크 단위 full-duplex 세션. 사용자 오디오를 밀어넣으면 Moshi 오디오가 나온다."""

    def __init__(self, model, processor, num_codebooks=8):
        self.model = model
        self.processor = processor
        self.mimi = processor.audio_tokenizer
        self.num_codebooks = num_codebooks
        self.reset()

    def reset(self):
        self.padding_cache = None          # Mimi conv 패딩 상태
        self.encoder_pkv = None            # Mimi encoder KV
        self.decoder_pkv = None            # Mimi decoder KV
        self.past_key_values = None        # Moshi KV (= 대화 문맥)
        self.first = True

    @torch.no_grad()
    def push(self, chunk: np.ndarray, **gen_kwargs):
        """chunk: (samples,) float32 @ 24kHz. 반환: (waveform, text)"""
        wav = torch.as_tensor(chunk, dtype=torch.float32, device=self.mimi.device)
        wav = wav.reshape(1, 1, -1)

        # 1) 사용자 오디오 -> 코드 (스트리밍 인코딩, 상태 되먹임)
        enc = self.mimi.encode(
            wav,
            num_quantizers=self.num_codebooks,
            encoder_past_key_values=self.encoder_pkv,
            padding_cache=self.padding_cache,
            use_streaming=True,
            return_dict=True,
        )
        self.encoder_pkv = enc.encoder_past_key_values
        self.padding_cache = enc.padding_cache
        user_codes = enc.audio_codes

        # 2) Moshi 한 청크 생성 — 대화 문맥은 past_key_values로 이어짐
        out = self.model.generate(
            user_audio_codes=user_codes,
            past_key_values=self.past_key_values,
            max_new_tokens=user_codes.shape[-1],   # 프레임 수만큼만 (full-duplex 정렬)
            concat_unconditional_inputs=self.first,
            return_audio_codes=True,
            return_dict_in_generate=True,
            use_cache=True,
            **gen_kwargs,
        )
        self.past_key_values = out.past_key_values
        self.first = False

        # 3) Moshi 코드 -> 파형. 주의: decode()는 encode()와 달리 use_streaming/padding_cache를
        # 받지 않는다 (audio_codes, padding_mask, decoder_past_key_values, return_dict).
        # 디코더 conv 상태는 청크 간 이어지지 않으므로 경계 아티팩트가 남을 수 있다.
        dec = self.mimi.decode(
            out.audio_codes,
            decoder_past_key_values=self.decoder_pkv,
            return_dict=True,
        )
        self.decoder_pkv = dec.decoder_past_key_values

        text = self.processor.tokenizer.decode(out.sequences[0], skip_special_tokens=True)
        return dec.audio_values[0, 0].float().cpu().numpy(), text

### 실행 — 80ms 청크로 밀어넣기

Mimi 프레임레이트가 12.5Hz면 1프레임 = 80ms. 청크를 프레임 경계에 맞춰야
인코더가 부분 프레임을 버리지 않습니다.

In [ ]:
samples_per_frame = int(sr / frame_rate)      # 24000 / 12.5 = 1920
frames_per_chunk = 5                          # 5프레임 = 400ms
chunk_samples = samples_per_frame * frames_per_chunk
print(f"{samples_per_frame} samples/frame, chunk = {chunk_samples} samples")

session = MoshiStreamingSession(model, processor)

# 실제로는 마이크 스트림. 여기서는 5초 무음을 청크로 쪼갠다.
stream = np.zeros(int(sr * 5.0), dtype=np.float32)
n_chunks = len(stream) // chunk_samples

audio_out, text_out = [], []
for i in range(n_chunks):
    chunk = stream[i * chunk_samples : (i + 1) * chunk_samples]
    wav, text = session.push(chunk, do_sample=True, temperature=0.8, top_k=250)
    audio_out.append(wav)
    if text.strip():
        text_out.append(text)
    print(f"chunk {i:2d}/{n_chunks}  out={wav.shape[0]:5d} samples  text={text!r}")

full = np.concatenate(audio_out)
print("\ntotal:", full.shape, full.shape[0] / sr, "sec")
print("transcript:", " ".join(text_out))

In [ ]:
Audio(full, rate=sr)

### 스트리밍이 제대로 붙었는지 확인

청크로 나눠 인코딩한 결과가 통짜 인코딩과 (거의) 같아야 합니다.
`padding_cache`를 되먹이지 않으면 여기서 어긋납니다.

In [ ]:
probe = np.random.RandomState(0).randn(chunk_samples * 4).astype(np.float32) * 0.1
probe_t = torch.as_tensor(probe, device=mimi.device).reshape(1, 1, -1)

with torch.no_grad():
    whole = mimi.encode(probe_t, num_quantizers=8).audio_codes

    pc, pkv, parts = None, None, []
    for i in range(4):
        seg = probe_t[..., i * chunk_samples : (i + 1) * chunk_samples]
        e = mimi.encode(
            seg, num_quantizers=8, encoder_past_key_values=pkv,
            padding_cache=pc, use_streaming=True, return_dict=True,
        )
        pkv, pc = e.encoder_past_key_values, e.padding_cache
        parts.append(e.audio_codes)
    chunked = torch.cat(parts, dim=-1)

print("whole  :", tuple(whole.shape))
print("chunked:", tuple(chunked.shape))
n = min(whole.shape[-1], chunked.shape[-1])
agree = (whole[..., :n] == chunked[..., :n]).float().mean().item()
print(f"code agreement: {agree:.1%}")

## 7. 학습 경로 (KD용)

`forward`는 `text_labels` / `audio_labels`를 받습니다. Depth Transformer 단독 클래스도 노출돼 있어
텍스트 스트림과 오디오 스트림 로짓을 분리해 distill할 수 있습니다.

In [ ]:
import inspect

from transformers import MoshiDepthDecoderForCausalLM  # noqa: F401

print(inspect.signature(MoshiForConditionalGeneration.forward))

with torch.no_grad():
    out = model(
        **inputs,
        text_labels=inputs["input_ids"],
        audio_labels=inputs["user_audio_codes"],
    )
print("loss:", out.loss)